**approval & rejection score**


•	Transaction type
•	Submission channel (Field Office / GOSI Online / Individual)
•	Establishment law classification (PPA, GOSI Gov, Semi-Gov, Private, etc.)
•	Establishment and contributor risk classification (White / Gray / Black list)
•	Historical approval and rejection patterns
•	Historical status (rejected txn or open txn)
•	Establishment violation history 
•	Completeness and quality of attached documents (OCR-based checks)
•	Data validation rules (e.g., backdated limits)
•	Specific Establishment / Contributor Historical data


- Transaction id = 101004
- Channel = taminaty (which is the individual channel)  
- also we should limit the MVP for GOSI law not PPA law establishments


In [225]:
import logging
import oracledb 
import yaml
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

load_dotenv()
import os

In [226]:
# # get config file for DB credentials and params
# try:
#     with open(r"C:\Users\059145\Workspace\config_am.yaml") as f:
#         config = yaml.safe_load(f)
#     logging.info("Configuration file loaded successfully")
# except Exception as e:
#     logging.error(f"Failed to load configuration: {e}")
#     raise 
#connect to Ameen USR Database

ameen_conn = oracledb.connect(user=os.getenv('AMEEN_USR'), password=os.getenv('AMEEN_PASS'), dsn=os.getenv('AMEEN_DSN'))
if ameen_conn.is_healthy():
    logging.info("Ameen USR connection successful!")
else:
    logging.error("Ameen USR connection failed")
    raise

fraud_conn = oracledb.connect(user=os.getenv('FRAUD_USR'), password=os.getenv('FRAUD_PASS'), dsn=os.getenv('FRAUD_DSN'))


In [140]:
def chunks(lst, n):
    """Yield successive n-sized chunks from lst"""
    for i in range(0, len(lst), n):
        yield lst[i:i + n]
 
def load_data(df_ids, SQL, conn):
    all_results = []
    
    for batch in chunks(df_ids.tolist(), 1000):
        # Build bind placeholders :1, :2, ..., :n
        placeholders = ','.join([f":{i+1}" for i in range(len(batch))])
        sql = f"""
        {SQL} IN ({placeholders})
        """
        # read_sql can take parameters for bind variables
        result_df = pd.read_sql(sql, conn, params=batch)
        all_results.append(result_df)
    
    # Combine all batches
    final_df = pd.concat(all_results, ignore_index=True)
    return final_df

In [24]:
query = """
-- SELECT * FROM T_TRANSACTIONACTIONREASON
with transaction as (
    select transactiontraceid, transactionid, establishmentid, personid  from T_TRANSACTIONTRACE
    -- where ROWNUM<10000
    where CREATIONTIMESTAMP >= to_date('01-03-2024','DD-MM-YYYY') and Channel='taminaty' and STATUS = 'Completed' and Transactionid = 101004
),
combined as (
select 
Actiondate,
step.STATUS, trans.*
from transaction trans left join 
T_TRANSACTIONSTEPTRACE step ON
trans.TRANSACTIONTRACEID = step.TRANSACTIONTRACEID
)
,
final_status as(
SELECT *
FROM (
    SELECT t.*,
           ROW_NUMBER() OVER (PARTITION BY TRANSACTIONTRACEID ORDER BY Actiondate DESC) AS rn
    FROM combined t
)
WHERE rn = 1

)


select *
from final_status
"""
df_status = pd.read_sql(query,con=ameen_conn)
# this takes a lot of time 

C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\157865249.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_status = pd.read_sql(query,con=ameen_conn)


In [42]:
df_status

,ACTIONDATE,STATUS,TRANSACTIONTRACEID,TRANSACTIONID,ESTABLISHMENTID,PERSONID,RN
0,2024-03-12 16:17:51,Rejected,240112034,101004,NaN,1.023218e+09,1
1,2024-04-23 07:12:36,Rejected,240236284,101004,NaN,1.039940e+09,1
2,2024-03-11 10:47:36,Rejected,240262765,101004,NaN,1.040212e+09,1
3,2024-04-22 16:34:37,Rejected,240302111,101004,NaN,1.029455e+09,1
4,2024-04-23 07:44:43,Rejected,240420020,101004,NaN,1.046360e+09,1
...,...,...,...,...,...,...,...
28043,2026-03-08 13:50:58,Rejected,889203590,101004,1.005238e+09,NaN,1
28044,2026-03-08 14:52:50,Rejected,889205048,101004,1.011989e+09,NaN,1
28045,2026-03-08 13:42:15,Approved,889223911,101004,1.011987e+09,NaN,1
28046,2026-03-08 14:08:49,Rejected,889227955,101004,1.011987e+09,NaN,1


In [26]:
df_status['STATUS'].value_counts()

STATUS
Rejected                13460
Approved                 9309
Expired                  5157
Inspection Completed       42
Unclaimed                  32
Submitted                  24
Assigned                   21
Withdrawn                   2
Reassigned                  1
Name: count, dtype: int64

**get the real establishmentid or personid**

In [27]:
qry = """
with parent as (
select transactiontraceid, transactionid from T_TRANSACTIONTRACE
where CREATIONTIMESTAMP >= to_date('01-03-2024','DD-MM-YYYY') and Channel='taminaty' and STATUS = 'Completed' and Transactionid = 101004
) 
-- paramkey, paramvalue 
select p.PARAMKEY,p.PARAMVALUE, p.transactiontraceid from T_TRANSACTIONTRACEPARAM p 
inner join parent 
on p.transactiontraceid = parent.transactiontraceid 
--where paramkey in ('RegistrationNo','NIN')
"""
df_traceparam = pd.read_sql(qry,con=ameen_conn)
df_traceparam

C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\2762032704.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_traceparam = pd.read_sql(qry,con=ameen_conn)


,PARAMKEY,PARAMVALUE,TRANSACTIONTRACEID
0,NIN,1098555087,240776053
1,ViolationRequestId,271375,240776053
2,ContributorViolationRequestId,1279170,240776053
3,SocialInsuranceNo,426421569,240508829
4,RegistrationNo,592274930,240508829
...,...,...,...
111934,ViolationRequestId,1409985,889227955
111935,ContributorViolationRequestId,1565880,889227955
111936,NIN,1060161153,889234942
111937,ViolationRequestId,1410017,889234942


In [28]:
# df_traceparam['TRANSACTIONTRACEID'].nunique()
# df_pivot

In [29]:
# Pivot so each PARAMKEY becomes a column
df_pivot = df_traceparam.pivot_table(
    index='TRANSACTIONTRACEID', 
    columns='PARAMKEY', 
    values='PARAMVALUE', 
    aggfunc='first'
).reset_index()

# Flatten column names
df_pivot.columns.name = None



In [54]:
df_merged = df_status.merge(df_pivot, on='TRANSACTIONTRACEID', how='inner')


In [55]:
df_merged

,ACTIONDATE,STATUS,TRANSACTIONTRACEID,TRANSACTIONID,ESTABLISHMENTID,PERSONID,RN,ContributorViolationRequestId,EngagementId,NIN,RegistrationNo,SocialInsuranceNo,ViolationRequestId
0,2024-03-12 16:17:51,Rejected,240112034,101004,NaN,1.023218e+09,1,1277964,NaN,1081062133,NaN,NaN,270280
1,2024-04-23 07:12:36,Rejected,240236284,101004,NaN,1.039940e+09,1,1277029,NaN,1124207117,NaN,NaN,270519
2,2024-03-11 10:47:36,Rejected,240262765,101004,NaN,1.040212e+09,1,1278760,NaN,1034501880,NaN,NaN,270564
3,2024-04-22 16:34:37,Rejected,240302111,101004,NaN,1.029455e+09,1,1277032,NaN,1102661244,NaN,NaN,270635
4,2024-04-23 07:44:43,Rejected,240420020,101004,NaN,1.046360e+09,1,1278176,NaN,1125822427,NaN,NaN,270859
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28043,2026-03-08 13:50:58,Rejected,889203590,101004,1.005238e+09,NaN,1,1565506,NaN,1111889943,NaN,NaN,1409791
28044,2026-03-08 14:52:50,Rejected,889205048,101004,1.011989e+09,NaN,1,1565507,NaN,1015397183,NaN,NaN,1409792
28045,2026-03-08 13:42:15,Approved,889223911,101004,1.011987e+09,NaN,1,1565878,1617663890,1069083432,645869729,401134069,1409979
28046,2026-03-08 14:08:49,Rejected,889227955,101004,1.011987e+09,NaN,1,1565880,NaN,1069083432,NaN,NaN,1409985


In [56]:
# df_merged['both_present'] = df_merged['ESTABLISHMENTID'].notna() & df_merged['RegistrationNo'].notna()

# df_merged['non_present'] = df_merged['ESTABLISHMENTID'].isna() & df_merged['RegistrationNo'].isna()

# df_merged['present_eng'] = df_merged['EngagementId'].notna() 

# df_merged['non_present_eng'] = df_merged['EngagementId'].isna() 

# df_merged['est_present'] = df_merged['ESTABLISHMENTID'].notna()
# df_merged['reg_present']= df_merged['RegistrationNo'].notna() 

In [ ]:
# This keeps NA as pandas’ native <NA> value and still behaves like an integer column
df_merged['ESTABLISHMENTID'] = df_merged['ESTABLISHMENTID'].astype('Int64')

In [47]:
# df_view1 = df_merged[df_merged['ESTABLISHMENTID'] == 1005238112]
# df_view1.columns

# to_drop = ['both_present', 'non_present', 'present_eng',
#        'non_present_eng']


# df_view1.drop(columns=to_drop, inplace=True)

In [48]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28048 entries, 0 to 28047
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   ACTIONDATE                     28048 non-null  datetime64[ns]
 1   STATUS                         28048 non-null  object        
 2   TRANSACTIONTRACEID             28048 non-null  int64         
 3   TRANSACTIONID                  28048 non-null  int64         
 4   ESTABLISHMENTID                19131 non-null  Int64         
 5   PERSONID                       14200 non-null  float64       
 6   RN                             28048 non-null  int64         
 7   ContributorViolationRequestId  28048 non-null  object        
 8   EngagementId                   9265 non-null   object        
 9   NIN                            28048 non-null  object        
 10  RegistrationNo                 9265 non-null   object        
 11  SocialInsurance

In [61]:
len(df_merged)

df_merged['ContributorViolationRequestId'] = df_merged['ContributorViolationRequestId'].astype(int)

In [51]:
# df_merged['reg_est_flag'] = np.where(
#     df_merged['reg_present'] | df_merged['est_present'],   # condition
#     1,                                         # value when True
#     0                                          # value when False
# )

# df_merged['reg_est_flag'].value_counts()

In [105]:
qry= """
-- NINMBER, GOSIREGISTRATIONNUMBER, CONTRIBUTORVIOLATIONREQID
SELECT  * FROM T_CONTRIBUTOR_VIOLATION_REQ 
WHERE CONTRIBUTORVIOLATIONREQID 
"""
# df_mix = pd.read_sql(qry,con=ameen_conn)

voilation_req_df = load_data(df_merged['ContributorViolationRequestId'].unique(), qry)

C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\702470785.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(sql, ameen_conn, params=batch)


In [106]:
voilation_req_df['CONTRIBUTORVIOLATIONREQID']

0        1286392
1        1285853
2        1286283
3        1286980
4        1287229
          ...   
28043    1565800
28044    1565846
28045    1565878
28046    1565880
28047    1565884
Name: CONTRIBUTORVIOLATIONREQID, Length: 28048, dtype: int64

In [107]:
df_merged['ContributorViolationRequestId']

0        1277964
1        1277029
2        1278760
3        1277032
4        1278176
          ...   
28043    1565506
28044    1565507
28045    1565878
28046    1565880
28047    1565884
Name: ContributorViolationRequestId, Length: 28048, dtype: int64

In [108]:
df_merged1 = df_merged.merge(
    voilation_req_df,
    left_on='ContributorViolationRequestId',
    right_on='CONTRIBUTORVIOLATIONREQID',
    how='inner'
)

In [109]:
df_est

,ESTABLISHMENTID,REGISTRATIONNUMBER,VERSIONNUMBER,ESTABLISHMENTNAMEARB,ESTABLISHMENTNAMEENG,ESTABLISHMENTSHORTNAMEARB,ESTABLISHMENTSHORTNAMEENG,FORMSUBMISSIONDATE,FORMSUBMISSIONDATEENTFMT,FIELDOFFICECODE,...,ESTSOURCETYPE,SOURCEESTID,DATAMASKEDEST,CLASS_SCALE,GCC_PROACTIVE,SPECIFICPAYMENTACCESSID,MCIENTITYTYPE,MCICOMPANYFORM,MCICRNCONFIRMDATE,MCICRNCONFIRMDATEENTFMT
0,5320500,10000602,1,الشركة السعودية للطاقة,None,None,None,1982-05-01,G,1,...,NaN,NaN,None,NaN,0,0,1.0,4.0,2026-06-09,G
1,85400,10008905,1,شركة الراشد للتجارة والمقاولات مساهمة مقفلة,None,None,None,1983-03-01,G,1,...,NaN,NaN,None,NaN,0,0,1.0,4.0,2025-11-20,G
2,232200,10044103,1,شركة زهران للصيانة والتشغيل,None,None,None,1983-05-13,G,1,...,NaN,NaN,None,NaN,0,0,1.0,4.0,2026-10-21,G
3,3806800,10071100,1,شركة الجهاز للمقاولات شركة شخص واحد,aljihaz Company For Contracting sharikat shakh...,None,None,1983-11-01,G,1,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-12-07,G
4,776600,13211477,1,شركة الفوزان للتجارة والمقاولات العامه,Alfawzan Company For Trading and Contracting a...,None,None,1985-08-01,G,1,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-08-27,G
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9190,1011988838,645885511,1,وزارة البيئة والمياه والزراعة,None,None,None,1933-10-21,H,1,...,1002.0,1230.0,None,0.0,0,0,NaN,NaN,NaT,None
9191,1012117408,647457819,1,مصنع عباية نواعم للصناعة,None,None,None,2018-02-01,G,10,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-10-27,G
9192,1012280648,649416699,1,شركة الصحة للموارد البشرية,None,None,None,2024-07-01,G,1,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-06-05,G
9193,1012338671,650111176,1,مؤسسة شذى عبدالرحيم مفرح المالكي للمقاولات,None,None,None,2024-09-01,G,12,...,NaN,NaN,None,NaN,0,0,NaN,NaN,NaT,None


In [ ]:
qry = """
SELECT * FROM T_ESTABLISHMENT
WHERE 
-- LAWTYPE = 1001 AND 
REGISTRATIONNUMBER
"""

df_est = load_data(df_merged1["GOSIREGISTRATIONNUMBER"].unique(), qry, ameen_conn)
# df_est = pd.read_sql(qry,con=ameen_conn)
# df_est

C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\702470785.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(sql, ameen_conn, params=batch)
C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\702470785.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(all_results, ignore_index=True)
C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\702470785.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determ

In [111]:
df_merged['reg_est_flag'] = np.where(
    df_merged['RegistrationNo'].notna()  | df_merged['ESTABLISHMENTID'].notna(),   # condition
    1,                                         # value when True
    0                                          # value when False
)

df_merged['reg_est_flag'].value_counts()

df_new = df_merged[df_merged['reg_est_flag'] == 1]
df_new

,ACTIONDATE,STATUS,TRANSACTIONTRACEID,TRANSACTIONID,ESTABLISHMENTID,PERSONID,RN,ContributorViolationRequestId,EngagementId,NIN,RegistrationNo,SocialInsuranceNo,ViolationRequestId,reg_est_flag
9,2024-03-10 12:47:06,Approved,240508829,101004,1.009037e+09,1.037884e+09,1,1279157,1599247444,1076345543,592274930,426421569,271012,1
11,2024-03-14 10:43:24,Approved,240587693,101004,1.011177e+09,1.019732e+09,1,1279351,1599359206,1085864310,634535322,385654383,271003,1
36,2024-03-13 11:26:10,Approved,241417508,101004,1.010739e+09,1.021499e+09,1,1280197,1599334052,1094009568,625160146,389515477,272714,1
39,2024-04-22 15:51:48,Approved,241509405,101004,1.000806e+09,1.015233e+09,1,1280603,1599358874,1040913228,503787113,377895827,272849,1
54,2024-03-17 14:37:36,Approved,241960045,101004,1.011798e+09,1.025552e+09,1,1281555,1600041302,1088692312,643681536,397279960,273687,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28043,2026-03-08 13:50:58,Rejected,889203590,101004,1.005238e+09,NaN,1,1565506,NaN,1111889943,NaN,NaN,1409791,1
28044,2026-03-08 14:52:50,Rejected,889205048,101004,1.011989e+09,NaN,1,1565507,NaN,1015397183,NaN,NaN,1409792,1
28045,2026-03-08 13:42:15,Approved,889223911,101004,1.011987e+09,NaN,1,1565878,1617663890,1069083432,645869729,401134069,1409979,1
28046,2026-03-08 14:08:49,Rejected,889227955,101004,1.011987e+09,NaN,1,1565880,NaN,1069083432,NaN,NaN,1409985,1


In [113]:
df_final0 = df_new.merge(
    df_est,
    on='ESTABLISHMENTID',
    how='inner'          # keep all df_new rows
)

In [115]:
# df_new['PERSONID'] = df_new['PERSONID'].astype('Int64')

In [116]:
# df_new = df_new.drop(columns=to_drop)

In [117]:
# df_new['NIN'].isnull().sum()

In [ ]:
df_new['ESTABLISHMENTID'].value_counts()

,ACTIONDATE,STATUS_x,TRANSACTIONTRACEID,TRANSACTIONID,ESTABLISHMENTID,PERSONID,RN,ContributorViolationRequestId,EngagementId,NIN,...,ESTSOURCETYPE,SOURCEESTID,DATAMASKEDEST,CLASS_SCALE,GCC_PROACTIVE,SPECIFICPAYMENTACCESSID,MCIENTITYTYPE,MCICOMPANYFORM,MCICRNCONFIRMDATE,MCICRNCONFIRMDATEENTFMT
0,2024-03-10 12:47:06,Approved,240508829,101004,1.009037e+09,1.037884e+09,1,1279157,1599247444,1076345543,...,NaN,NaN,None,NaN,0,0,1.0,4.0,2026-04-06,G
1,2024-03-14 10:43:24,Approved,240587693,101004,1.011177e+09,1.019732e+09,1,1279351,1599359206,1085864310,...,NaN,NaN,None,NaN,0,0,4.0,NaN,2026-07-11,G
2,2024-03-13 11:26:10,Approved,241417508,101004,1.010739e+09,1.021499e+09,1,1280197,1599334052,1094009568,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-11-01,G
3,2024-04-22 15:51:48,Approved,241509405,101004,1.000806e+09,1.015233e+09,1,1280603,1599358874,1040913228,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-09-18,G
4,2024-03-17 14:37:36,Approved,241960045,101004,1.011798e+09,1.025552e+09,1,1281555,1600041302,1088692312,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-08-12,G
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19126,2026-03-08 13:50:58,Rejected,889203590,101004,1.005238e+09,NaN,1,1565506,NaN,1111889943,...,NaN,NaN,None,NaN,0,0,NaN,NaN,2026-10-23,G
19127,2026-03-08 14:52:50,Rejected,889205048,101004,1.011989e+09,NaN,1,1565507,NaN,1015397183,...,1002.0,1230.0,None,0.0,0,0,NaN,NaN,NaT,None
19128,2026-03-08 13:42:15,Approved,889223911,101004,1.011987e+09,NaN,1,1565878,1617663890,1069083432,...,1002.0,591.0,None,0.0,0,0,NaN,NaN,NaT,None
19129,2026-03-08 14:08:49,Rejected,889227955,101004,1.011987e+09,NaN,1,1565880,NaN,1069083432,...,1002.0,591.0,None,0.0,0,0,NaN,NaN,NaT,None


In [147]:
approved_labels = {'approved'}          # a set – easy to extend if needed
df_final0['is_approved'] = df_final0['STATUS_x'].str.lower().isin(approved_labels)

In [120]:
df_new['is_approved']

9         True
11        True
36        True
39        True
54        True
         ...  
28043    False
28044    False
28045     True
28046    False
28047    False
Name: is_approved, Length: 19264, dtype: bool

In [121]:
approval_per_est = (
    df_new.groupby('ESTABLISHMENTID')
      .agg(
          total_requests=('is_approved', 'size'),          # count rows
          approved_requests=('is_approved', 'sum')         # count Trues
      )
      .reset_index()
)

In [148]:
approval_per_est

,ESTABLISHMENTID,total_requests,approved_requests,approval_percent
0,1.011987e+09,1809,1678,92.76
1,1.011987e+09,1394,1272,91.25
2,1.011987e+09,1215,329,27.08
3,1.011988e+09,1015,981,96.65
4,1.011987e+09,833,369,44.30
...,...,...,...,...
2321,1.013035e+09,1,0,0.00
2322,1.013045e+09,1,0,0.00
2323,1.013073e+09,1,0,0.00
2324,1.013099e+09,1,0,0.00


In [149]:

approval_per_est['approval_percent'] = (
    approval_per_est['approved_requests'] /
    approval_per_est['total_requests'] * 100
).round(2)

In [150]:
approval_per_est

,ESTABLISHMENTID,total_requests,approved_requests,approval_percent
0,1.011987e+09,1809,1678,92.76
1,1.011987e+09,1394,1272,91.25
2,1.011987e+09,1215,329,27.08
3,1.011988e+09,1015,981,96.65
4,1.011987e+09,833,369,44.30
...,...,...,...,...
2321,1.013035e+09,1,0,0.00
2322,1.013045e+09,1,0,0.00
2323,1.013073e+09,1,0,0.00
2324,1.013099e+09,1,0,0.00


In [151]:
approval_per_est = approval_per_est.sort_values(
    by='total_requests',   # column to sort on
    ascending=False,         # False → descending (largest → smallest)
    kind='mergesort'         # stable sort (keeps original order for ties)
).reset_index(drop=True) 

In [152]:
# df_new

In [153]:
df_final = df_final0.merge(
    approval_per_est,
    on='ESTABLISHMENTID',
    how='left'          # keep all df_new rows
)

In [154]:
df_final

,ACTIONDATE,STATUS_x,TRANSACTIONTRACEID,TRANSACTIONID,ESTABLISHMENTID,PERSONID,RN,ContributorViolationRequestId,EngagementId,NIN,...,GCC_PROACTIVE,SPECIFICPAYMENTACCESSID,MCIENTITYTYPE,MCICOMPANYFORM,MCICRNCONFIRMDATE,MCICRNCONFIRMDATEENTFMT,is_approved,total_requests,approved_requests,approval_percent
0,2024-03-10 12:47:06,Approved,240508829,101004,1.009037e+09,1.037884e+09,1,1279157,1599247444,1076345543,...,0,0,1.0,4.0,2026-04-06,G,True,1,1,100.00
1,2024-03-14 10:43:24,Approved,240587693,101004,1.011177e+09,1.019732e+09,1,1279351,1599359206,1085864310,...,0,0,4.0,NaN,2026-07-11,G,True,1,1,100.00
2,2024-03-13 11:26:10,Approved,241417508,101004,1.010739e+09,1.021499e+09,1,1280197,1599334052,1094009568,...,0,0,NaN,NaN,2026-11-01,G,True,1,1,100.00
3,2024-04-22 15:51:48,Approved,241509405,101004,1.000806e+09,1.015233e+09,1,1280603,1599358874,1040913228,...,0,0,NaN,NaN,2026-09-18,G,True,1,1,100.00
4,2024-03-17 14:37:36,Approved,241960045,101004,1.011798e+09,1.025552e+09,1,1281555,1600041302,1088692312,...,0,0,NaN,NaN,2026-08-12,G,True,1,1,100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19126,2026-03-08 13:50:58,Rejected,889203590,101004,1.005238e+09,NaN,1,1565506,NaN,1111889943,...,0,0,NaN,NaN,2026-10-23,G,False,4,0,0.00
19127,2026-03-08 14:52:50,Rejected,889205048,101004,1.011989e+09,NaN,1,1565507,NaN,1015397183,...,0,0,NaN,NaN,NaT,None,False,199,127,63.82
19128,2026-03-08 13:42:15,Approved,889223911,101004,1.011987e+09,NaN,1,1565878,1617663890,1069083432,...,0,0,NaN,NaN,NaT,None,True,151,90,59.60
19129,2026-03-08 14:08:49,Rejected,889227955,101004,1.011987e+09,NaN,1,1565880,NaN,1069083432,...,0,0,NaN,NaN,NaT,None,False,151,90,59.60


In [155]:
df_final = df_final.sort_values(
    by='total_requests',   # column to sort on
    ascending=False,         # False → descending (largest → smallest)
    kind='mergesort'         # stable sort (keeps original order for ties)
).reset_index(drop=True) 

In [156]:
df_final

,ACTIONDATE,STATUS_x,TRANSACTIONTRACEID,TRANSACTIONID,ESTABLISHMENTID,PERSONID,RN,ContributorViolationRequestId,EngagementId,NIN,...,GCC_PROACTIVE,SPECIFICPAYMENTACCESSID,MCIENTITYTYPE,MCICOMPANYFORM,MCICRNCONFIRMDATE,MCICRNCONFIRMDATEENTFMT,is_approved,total_requests,approved_requests,approval_percent
0,2024-07-11 11:38:09,Approved,260909303,101004,1.011987e+09,1.044328e+09,1,1293672,1602092946,1044911475,...,0,0,NaN,NaN,NaT,None,True,1809,1678,92.76
1,2024-07-09 09:11:50,Approved,261139122,101004,1.011987e+09,1.020896e+09,1,1294544,1602020731,1061207781,...,0,0,NaN,NaN,NaT,None,True,1809,1678,92.76
2,2024-07-09 13:32:03,Approved,261410232,101004,1.011987e+09,1.047037e+09,1,1296105,1602028912,1001418795,...,0,0,NaN,NaN,NaT,None,True,1809,1678,92.76
3,2024-07-11 10:50:14,Approved,261675606,101004,1.011987e+09,1.024893e+09,1,1296852,1602091615,1094530357,...,0,0,NaN,NaN,NaT,None,True,1809,1678,92.76
4,2024-07-14 08:56:13,Approved,262476156,101004,1.011987e+09,1.047037e+09,1,1298297,1602156702,1001418795,...,0,0,NaN,NaN,NaT,None,True,1809,1678,92.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19126,2026-03-04 10:55:11,Rejected,888608244,101004,1.010749e+09,NaN,1,1560898,NaN,1030086308,...,0,0,NaN,NaN,2026-09-30,G,False,1,0,0.00
19127,2026-03-04 12:37:54,Rejected,888622429,101004,1.010833e+09,NaN,1,1559589,NaN,1063120362,...,0,0,NaN,NaN,2026-06-02,G,False,1,0,0.00
19128,2026-03-05 13:01:38,Rejected,888865888,101004,1.012339e+09,NaN,1,1559665,NaN,1098670688,...,0,0,NaN,NaN,NaT,None,False,1,0,0.00
19129,2026-03-08 11:29:54,Rejected,889063455,101004,1.012117e+09,NaN,1,1565475,NaN,1093530697,...,0,0,NaN,NaN,2026-10-27,G,False,1,0,0.00


In [157]:
# df_merged['both_present'] = df_merged['ESTABLISHMENTID'].notna() & df_merged['RegistrationNo'].notna()

# df_merged['non_present'] = df_merged['ESTABLISHMENTID'].isna() & df_merged['RegistrationNo'].isna()

df_merged['present_voi'] = df_merged['ViolationRequestId'].notna() 

df_merged['non_present_voi'] = df_merged['ViolationRequestId'].isna() 


df_merged.groupby('STATUS').agg(overlap_count=('present_voi', 'sum'),none_count=('non_present_voi', 'sum')).reset_index()

,STATUS,overlap_count,none_count
0,Approved,9309,0
1,Assigned,21,0
2,Expired,5157,0
3,Inspection Completed,42,0
4,Reassigned,1,0
5,Rejected,13460,0
6,Submitted,24,0
7,Unclaimed,32,0
8,Withdrawn,2,0


In [159]:
df_final['present_NIN'] = df_final['NIN'].notna() 

df_final['non_present_NIN'] = df_final['NIN'].isna() 


df_final.groupby('STATUS_x').agg(overlap_count=('present_NIN', 'sum'),none_count=('non_present_NIN', 'sum')).reset_index()

,STATUS_x,overlap_count,none_count
0,Approved,9117,0
1,Assigned,17,0
2,Expired,4695,0
3,Inspection Completed,37,0
4,Reassigned,1,0
5,Rejected,5237,0
6,Submitted,6,0
7,Unclaimed,19,0
8,Withdrawn,2,0


In [221]:
filtered_df = df_final[df_final['STATUS_x'].isin(['Rejected', 'Approved'])]

In [222]:
# filtered_df


In [223]:
# filtered_df['ESTABLISHMENTID'].unique

#   filtered_df["ESTABLISHMENTID"].unqiue
# filtered_df['ESTABLISHMENTID'].unique()

In [228]:
sql = "select * from FRAUD_USR.T_EST_ENC_MAP where ESTABLISHMENTID "

masking_table = load_data(filtered_df['ESTABLISHMENTID'].unique(), sql, fraud_conn)

C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\2785801832.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(sql, conn, params=batch)


In [229]:
filtered_df = filtered_df.merge(masking_table, on='ESTABLISHMENTID')

In [230]:
sql = "select ESTABLISHMENTID,CREATION_DATE,COMPOUND_VALUE_1 from FRAUD_USR.KASHIF_INDICATORS where ESTABLISHMENTID "

kashef_indicators = load_data(masking_table['ESTABLISHMENTID_ENC'].unique(),sql,fraud_conn)

kashef_indicators = kashef_indicators.sort_values('CREATION_DATE', ascending=False).drop_duplicates(subset='ESTABLISHMENTID', keep='first')


C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\2785801832.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(sql, conn, params=batch)


In [231]:
kashef_indicators.groupby('ESTABLISHMENTID').count()

,CREATION_DATE,COMPOUND_VALUE_1
ESTABLISHMENTID,,
3668,1,1
1960868,1,1
2530868,1,1
2788868,1,1
5208068,1,1
...,...,...
12151312160,1,1
12151393664,1,1
12151601624,1,1


In [259]:
filtered_df = filtered_df[filtered_df['LAWTYPE']==1001]

In [260]:
filtered_df = filtered_df.merge(kashef_indicators,     
                                left_on='ESTABLISHMENTID_ENC',
                                right_on='ESTABLISHMENTID',
                                how='left')

In [272]:
# df_clean[df_clean['LAWTYPE']==1001].groupby('STATUS_x').count()
# df_clean

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df_clean = filtered_df.dropna(subset=['COMPOUND_VALUE_1'])

# Example DataFrame
# df = pd.DataFrame({'status': ['approved', 'rejected', ...], 'score': [75, 60, ...]})

sns.boxplot(x='STATUS_x', y='COMPOUND_VALUE_1', data=df_clean)

# Group by binary column
summary = df_clean.groupby('STATUS_x')['COMPOUND_VALUE_1'].describe()
print(summary[['min', '25%', '50%', '75%', 'max']])

plt.show()

sns.violinplot(x='STATUS_x', y='COMPOUND_VALUE_1', data=df_clean)
plt.title("Violin Plot of Score by Status")
plt.show()

sns.stripplot(x='STATUS_x', y='COMPOUND_VALUE_1', data=df_clean, jitter=True, alpha=0.5)
plt.title("Strip Plot of Score by Status")
plt.show()

grouped = df_clean.groupby('STATUS_x')['COMPOUND_VALUE_1']
summary_stats = grouped.agg(['min', 'quantile', 'median', 'max', 'mean'])
print(summary_stats)

KeyError: ['COMPOUND_VALUE_1']

In [263]:
from scipy.stats import ttest_ind

approved = df_clean[df_clean['STATUS_x'] == 'Approved']['COMPOUND_VALUE_1']
rejected = df_clean[df_clean['STATUS_x'] == 'Rejected']['COMPOUND_VALUE_1']

t_stat, p_value = ttest_ind(approved, rejected)
print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 32.040861150584405
p-value: 1.5980116765359741e-217


In [264]:
# !pip install statsmodels

In [265]:
import statsmodels.api as sm

# Encode binary variable as 0/1
df_clean['status_binary'] = df_clean['STATUS_x'].map({'Rejected': 0, 'Approved': 1})

X = df_clean[['COMPOUND_VALUE_1']]  # predictor
X = sm.add_constant(X)  # adds intercept term
y = df_clean['status_binary']

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.618574
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:          status_binary   No. Observations:                14145
Model:                          Logit   Df Residuals:                    14143
Method:                           MLE   Df Model:                            1
Date:                Thu, 12 Mar 2026   Pseudo R-squ.:                 0.05189
Time:                        11:04:56   Log-Likelihood:                -8749.7
converged:                       True   LL-Null:                       -9228.6
Covariance Type:            nonrobust   LLR p-value:                2.809e-210
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -4.7149      0.206    -22.932      0.000      -5.118      -4.312
COMPOUND_VA

C:\Users\CR241121\AppData\Local\Temp\ipykernel_65132\2735824449.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['status_binary'] = df_clean['STATUS_x'].map({'Rejected': 0, 'Approved': 1})


In [266]:
from scipy.stats import pointbiserialr

r, p = pointbiserialr(df_clean['status_binary'], df_clean['COMPOUND_VALUE_1'])
print("Correlation:", r)
print("p-value:", p)

Correlation: 0.2601458337765478
p-value: 1.5980116765341535e-217


In [267]:

# df_merged.groupby('STATUS').agg(
#     establishment_count=('ESTABLISHMENTID', 'count'),
#     registration_count=('RegistrationNo', 'count'),
#     overlap_count=('both_present', 'sum'),
#     none_count=('non_present', 'sum')
# ).reset_index()

In [268]:
df_view = df_merged[df_merged['RegistrationNo'].isna()]